# Task 4 — Open-Set Recognition (Drive-native Colab)

**Intended workflow**
1. Open this notebook from **the same Google Drive** that already has `ATML-PA1`
   (Drive file browser → open `.ipynb`, or upload this file into that Drive).
2. Runtime → **GPU (T4)**.
3. Mount Drive → `cd` into the existing repo → run stages.
4. All outputs write into `ATML-PA1/task4/results/` on Drive (durable).

No GitHub clone required when the code is already on this Drive.

**Hard rules:** CIFAR-10 val Acc for checkpoints; CIFAR-10 val only for thresholds; no CIFAR-100 in train.

In [ ]:
import torch

print("cuda:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Runtime → Change runtime type → T4 GPU, then re-run.")
print("device:", torch.cuda.get_device_name(0))

## 1) Mount Drive and enter the existing repo

Edit `REPO_DIR` only if your folder path differs.

In [ ]:
from pathlib import Path
import os
from google.colab import drive

drive.mount("/content/drive")

# Path on THIS Drive (same account that owns the code)
REPO_DIR = Path("/content/drive/MyDrive/MS AI/Semester_3/ATML/PAs/ATML-PA1")

if not (REPO_DIR / "task4" / "train.py").is_file():
    alt = Path("/content/drive/MyDrive/ATML-PA1")
    if (alt / "task4" / "train.py").is_file():
        REPO_DIR = alt
    else:
        raise SystemExit(
            f"Repo not found at {REPO_DIR}. Set REPO_DIR to your ATML-PA1 folder."
        )

os.chdir(REPO_DIR)
RESULTS = REPO_DIR / "task4" / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
BACKUP = Path("/content/drive/MyDrive/ATML-PA1-task4-backup")
BACKUP.mkdir(parents=True, exist_ok=True)

print("cwd:", Path.cwd())
print("task4 OK:", (REPO_DIR / "task4" / "train.py").is_file())
print("results →", RESULTS)
print("ckpts:", sorted(p.name for p in (RESULTS / "checkpoints").glob("*.pt")) if (RESULTS / "checkpoints").exists() else [])

In [ ]:
%pip install -q -r requirements.txt

## 2) Helpers — refresh zip on Drive after each stage

In [ ]:
from pathlib import Path
import shutil

REPO_DIR = Path.cwd()
RESULTS = REPO_DIR / "task4" / "results"
BACKUP = Path("/content/drive/MyDrive/ATML-PA1-task4-backup")
BACKUP.mkdir(parents=True, exist_ok=True)

def refresh_bundle(tag: str = ""):
    """Rebuild zip in repo root + backup folder; copy pipeline log."""
    RESULTS.mkdir(parents=True, exist_ok=True)
    for zip_base in (BACKUP / "task4_results_bundle", REPO_DIR / "task4_results_bundle"):
        zp = zip_base.with_suffix(".zip")
        if zp.exists():
            zp.unlink()
        shutil.make_archive(str(zip_base), "zip", root_dir=RESULTS)
        print(f"[{tag}] {zp} ({zp.stat().st_size} bytes)")
    log = Path("/content/task4_pipeline.log")
    if log.exists():
        shutil.copy2(log, BACKUP / "task4_pipeline.log")
        shutil.copy2(log, REPO_DIR / "task4_pipeline.log")
    for sub in ["checkpoints", "tables", "curves", "figures", "splits", "cache"]:
        p = RESULTS / sub
        n = sum(1 for _ in p.rglob("*") if _.is_file()) if p.exists() else 0
        print(f"  {sub}: {n} files")

refresh_bundle("init")

## 3) Start pipeline (background, resume-safe)

Skips train stages whose `*_best.pt` already exists under `task4/results/checkpoints/`.
Refreshes the Drive zip after every stage so you can leave it unattended.

In [ ]:
import os, subprocess
from pathlib import Path

REPO_DIR = Path.cwd()
assert (REPO_DIR / "task4" / "train.py").is_file(), "cd into ATML-PA1 first (cell above)"

log = Path("/content/task4_pipeline.log")
done = Path("/content/task4_pipeline.done")
if done.exists():
    done.unlink()
log.write_text("starting Drive-native pipeline\n")

# Use absolute repo path inside the shell script
repo = str(REPO_DIR)

script = f'''
set -e
cd "{repo}"
CKPT=task4/results/checkpoints
mkdir -p "$CKPT"
BACKUP=/content/drive/MyDrive/ATML-PA1-task4-backup
mkdir -p "$BACKUP"

refresh() {{
  TAG="$1"
  python - <<PY
from pathlib import Path
import shutil
REPO = Path(r"{repo}")
RESULTS = REPO / "task4" / "results"
BACKUP = Path("$BACKUP")
for zip_base in (BACKUP / "task4_results_bundle", REPO / "task4_results_bundle"):
    zp = zip_base.with_suffix(".zip")
    if zp.exists():
        zp.unlink()
    shutil.make_archive(str(zip_base), "zip", root_dir=RESULTS)
    print("[", "$TAG", "]", zp, zp.stat().st_size)
log = Path("/content/task4_pipeline.log")
if log.exists():
    shutil.copy2(log, BACKUP / "task4_pipeline.log")
    shutil.copy2(log, REPO / "task4_pipeline.log")
for sub in ["checkpoints", "tables", "curves", "figures", "splits", "cache"]:
    p = RESULTS / sub
    n = sum(1 for _ in p.rglob("*") if _.is_file()) if p.exists() else 0
    print(f"  {{sub}}: {{n}} files")
PY
}}

python -m task4.scripts.run_task4 --stages splits
refresh splits

if [ ! -f "$CKPT/vanilla_best.pt" ]; then
  python -m task4.scripts.run_task4 --stages train_vanilla
  refresh vanilla
else
  echo "SKIP train_vanilla"
fi

if [ ! -f "$CKPT/gcsc_best.pt" ]; then
  python -m task4.scripts.run_task4 --stages train_gcsc
  refresh gcsc
else
  echo "SKIP train_gcsc"
fi

if [ ! -f "$CKPT/proser_best.pt" ]; then
  python -m task4.scripts.run_task4 --stages train_proser
  refresh proser
else
  echo "SKIP train_proser"
fi

python -m task4.scripts.run_task4 --stages extract_all
refresh extract

python -m task4.scripts.run_task4 --stages eval
refresh eval

touch /content/task4_pipeline.done
cp /content/task4_pipeline.done "$BACKUP/" || true
cp /content/task4_pipeline.done "{repo}/" || true
echo DONE
'''

proc = subprocess.Popen(
    ["bash", "-lc", script],
    stdout=open(log, "a"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print("PID", proc.pid)
print("Running from:", REPO_DIR)
print("Checkpoints →", REPO_DIR / "task4/results/checkpoints")
print("Zip also →", Path("/content/drive/MyDrive/ATML-PA1-task4-backup/task4_results_bundle.zip"))
print("Poll with the next cell. Safe to leave unattended.")

## 4) Poll progress (re-run anytime)

In [ ]:
from pathlib import Path
import subprocess

log = Path("/content/task4_pipeline.log")
done = Path("/content/task4_pipeline.done")
RESULTS = Path.cwd() / "task4" / "results"
print("done", done.exists() or (Path.cwd() / "task4_pipeline.done").exists())
r = subprocess.run(["bash", "-lc", "pgrep -af 'python3 -m task4' || true"], capture_output=True, text=True)
print("procs:\n", r.stdout)

if log.exists():
    lines = log.read_text(errors="ignore").splitlines()
    keep = [
        ln for ln in lines
        if ("epoch" in ln.lower() or ">>" in ln or "best" in ln or "SKIP" in ln
            or "zip" in ln.lower() or "DONE" in ln or "Error" in ln or "Traceback" in ln
            or "files" in ln or "Wrote" in ln)
        and "it/s" not in ln
    ]
    print("--- tail ---")
    print("\n".join(keep[-40:]))

print("\ncheckpoints:")
ck = RESULTS / "checkpoints"
if ck.exists():
    for p in sorted(ck.glob("*.pt")):
        print(f"  {p.name}  {p.stat().st_size} B")
else:
    print("  (none yet)")

zp = Path("/content/drive/MyDrive/ATML-PA1-task4-backup/task4_results_bundle.zip")
print("backup zip:", zp.exists(), zp.stat().st_size if zp.exists() else "")

## Done checklist

On Drive you should have:
- `ATML-PA1/task4/results/checkpoints/{vanilla,gcsc,proser}_best.pt`
- `ATML-PA1/task4/results/tables/*.json`
- `ATML-PA1/task4/results/figures/score_distributions.png`
- `ATML-PA1/task4_results_bundle.zip`
- `MyDrive/ATML-PA1-task4-backup/task4_results_bundle.zip` (mirror)

If runtime dies: reopen this notebook → GPU → mount → start pipeline again (finished ckpts are skipped).